# WESAD HRV Stress Detection — Experiment Notebook

Produces the **real results** for the paper *"Generalizable Wearable Stress Detection
from Heart-Rate Variability."*

**Pipeline:** download WESAD -> extract HRV features from wrist PPG -> Leave-One-Subject-Out
(LOSO) evaluation of baseline vs stress -> ablations -> figures.

**How to run:** open in Google Colab (Runtime -> Run all). A free CPU runtime is fine;
the WESAD download is ~2 GB and runs in the cloud, so your laptop storage is irrelevant.

> **Data caveat:** the WESAD download URL below is the commonly used public link. If it
> 404s or changes, get the current link from the official WESAD page
> (search "WESAD dataset Uni Siegen ICMI18") and replace `WESAD_URL`.


In [ ]:
# 1. Dependencies
!pip -q install neurokit2==0.2.10 scikit-learn scipy numpy pandas matplotlib tqdm

In [ ]:
# 2. Download + unzip WESAD (~2 GB, in Colab)
import os, zipfile
DATA_DIR = "/content/wesad"
ZIP_PATH = "/content/WESAD.zip"
WESAD_URL = "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"  # replace if it fails
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATA_DIR, "WESAD", "S2", "S2.pkl")):
    if not os.path.exists(ZIP_PATH):
        print("Downloading WESAD (~2 GB)...")
        !wget -q -O "$ZIP_PATH" "$WESAD_URL"
    print("Unzipping...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
print("Subjects:", sorted(os.listdir(os.path.join(DATA_DIR, "WESAD"))))

In [ ]:
# 3. Imports + configuration
import pickle, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import neurokit2 as nk
from scipy import signal as spsig, interpolate as spinterp
from tqdm import tqdm

WESAD_ROOT = os.path.join(DATA_DIR, "WESAD")
SUBJECTS   = [f"S{i}" for i in [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]]  # WESAD has no S1/S12
BVP_FS     = 64      # wrist PPG sampling rate (Hz)
LABEL_FS   = 700     # label stream sampling rate (Hz)
WINDOW_SEC = 60
STRIDE_SEC = 30
LABEL_MAP  = {1: 0, 2: 1}   # 1=baseline -> non-stress(0), 2=stress -> stress(1)
PURITY     = 0.90           # window must be >=90% one class
MIN_BEATS  = 12
OUT_DIR    = "/content/outputs"; os.makedirs(OUT_DIR, exist_ok=True)
FEATURES   = ["meanHR","SDNN","RMSSD","pNN50","LF","HF","LFHF"]

In [ ]:
# 4. HRV feature extraction from a PPG window
def freq_domain(rri_ms):
    """LF/HF from an RR-interval series (ms) via interpolation + Welch PSD."""
    t = np.cumsum(rri_ms) / 1000.0
    t = t - t[0]
    if t[-1] < 20 or len(rri_ms) < 8:
        return np.nan, np.nan, np.nan
    fs_i = 4.0
    ti = np.arange(0, t[-1], 1/fs_i)
    rri_i = spinterp.interp1d(t, rri_ms, kind="cubic", fill_value="extrapolate")(ti)
    rri_i = rri_i - np.mean(rri_i)
    nperseg = min(len(rri_i), 256)
    freqs, psd = spsig.welch(rri_i, fs=fs_i, nperseg=nperseg)
    lf = np.trapz(psd[(freqs>=0.04)&(freqs<0.15)], freqs[(freqs>=0.04)&(freqs<0.15)])
    hf = np.trapz(psd[(freqs>=0.15)&(freqs<0.40)], freqs[(freqs>=0.15)&(freqs<0.40)])
    return lf, hf, (lf/hf if hf > 0 else np.nan)

def hrv_features(bvp):
    """Return dict of HRV features for one PPG window, or None if unusable."""
    try:
        cleaned = nk.ppg_clean(bvp, sampling_rate=BVP_FS)
        peaks = nk.ppg_findpeaks(cleaned, sampling_rate=BVP_FS)["PPG_Peaks"]
    except Exception:
        return None
    if len(peaks) < MIN_BEATS:
        return None
    rri = np.diff(peaks) / BVP_FS * 1000.0
    rri = rri[(rri >= 300) & (rri <= 2000)]   # keep physiologically plausible beats
    if len(rri) < MIN_BEATS - 1:
        return None
    d = np.diff(rri)
    lf, hf, lfhf = freq_domain(rri)
    return dict(meanHR=60000.0/np.mean(rri), SDNN=np.std(rri, ddof=1),
                RMSSD=np.sqrt(np.mean(d**2)), pNN50=np.mean(np.abs(d) > 50)*100,
                LF=lf, HF=hf, LFHF=lfhf)

In [ ]:
# 5. Window each subject and build the feature table
def build_features(window_sec=WINDOW_SEC, stride_sec=STRIDE_SEC):
    rows = []
    for subj in tqdm(SUBJECTS, desc=f"win={window_sec}s"):
        with open(os.path.join(WESAD_ROOT, subj, f"{subj}.pkl"), "rb") as fh:
            data = pickle.load(fh, encoding="latin1")
        bvp   = np.asarray(data["signal"]["wrist"]["BVP"]).flatten()
        label = np.asarray(data["label"]).flatten()
        wlen, stride = window_sec*BVP_FS, stride_sec*BVP_FS
        for i in range(0, len(bvp)-wlen, stride):
            t0, t1 = i/BVP_FS, (i+wlen)/BVP_FS
            lab = label[int(t0*LABEL_FS):int(t1*LABEL_FS)]
            if len(lab) == 0:
                continue
            vals, counts = np.unique(lab, return_counts=True)
            dom = vals[np.argmax(counts)]
            if dom not in LABEL_MAP or counts.max()/counts.sum() < PURITY:
                continue
            feats = hrv_features(bvp[i:i+wlen])
            if feats is None:
                continue
            feats["label"], feats["subject"] = LABEL_MAP[dom], subj
            rows.append(feats)
    return pd.DataFrame(rows).dropna().reset_index(drop=True)

df = build_features()
df.to_csv(os.path.join(OUT_DIR, "features_w60.csv"), index=False)
print("Feature table:", df.shape)
print(df["label"].value_counts().rename({0:"non-stress",1:"stress"}))
print("\nWindows per subject:\n", df.groupby("subject")["label"].count())

In [ ]:
# 6. Leave-One-Subject-Out evaluation
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def loso_eval(df, feature_cols, model_fn):
    accs, f1s, aucs = [], [], []
    for test_subj in SUBJECTS:
        tr = df[df.subject != test_subj]; te = df[df.subject == test_subj]
        if te.empty or tr.label.nunique() < 2:
            continue
        sc  = StandardScaler().fit(tr[feature_cols])
        clf = model_fn().fit(sc.transform(tr[feature_cols]), tr.label)
        pred = clf.predict(sc.transform(te[feature_cols]))
        accs.append(accuracy_score(te.label, pred))
        f1s.append(f1_score(te.label, pred, average="macro", zero_division=0))
        if te.label.nunique() == 2 and hasattr(clf, "predict_proba"):
            aucs.append(roc_auc_score(te.label, clf.predict_proba(sc.transform(te[feature_cols]))[:,1]))
    return dict(acc=np.mean(accs), f1=np.mean(f1s), auc=(np.mean(aucs) if aucs else np.nan),
                f1_std=np.std(f1s), n_folds=len(f1s), f1_list=f1s)

MODELS = {
    "meanHR-only (baseline)": (["meanHR"], lambda: LogisticRegression(max_iter=1000)),
    "Logistic Regression":    (FEATURES,  lambda: LogisticRegression(max_iter=1000)),
    "Random Forest":          (FEATURES,  lambda: RandomForestClassifier(n_estimators=300, random_state=42)),
    "Small MLP":              (FEATURES,  lambda: MLPClassifier(hidden_layer_sizes=(32,16), max_iter=500, random_state=42)),
}
rows = []
per_model_f1 = {}
for name,(cols,fn) in MODELS.items():
    r = loso_eval(df, cols, fn); per_model_f1[name] = r.pop("f1_list"); r["model"] = name
    rows.append(r)
results = pd.DataFrame(rows)[["model","acc","f1","auc","f1_std","n_folds"]].round(3)
results.to_csv(os.path.join(OUT_DIR, "results_main.csv"), index=False)
results

In [ ]:
# 7a. Ablation: feature groups (time vs frequency vs all) with Random Forest
groups = {"time-only":["meanHR","SDNN","RMSSD","pNN50"], "freq-only":["LF","HF","LFHF"], "all":FEATURES}
ab = []
for g, cols in groups.items():
    r = loso_eval(df, cols, lambda: RandomForestClassifier(n_estimators=300, random_state=42))
    r.pop("f1_list"); r["features"] = g; ab.append(r)
ablation_feat = pd.DataFrame(ab)[["features","acc","f1","auc","f1_std"]].round(3)
ablation_feat.to_csv(os.path.join(OUT_DIR, "ablation_features.csv"), index=False)
ablation_feat

In [ ]:
# 7b. Ablation: window length (re-extracts features; takes a few minutes)
win_rows = []
for w in [30, 60, 120]:
    dfw = build_features(window_sec=w, stride_sec=max(15, w//2))
    r = loso_eval(dfw, FEATURES, lambda: RandomForestClassifier(n_estimators=300, random_state=42))
    r.pop("f1_list"); r["window_sec"] = w; win_rows.append(r)
ablation_window = pd.DataFrame(win_rows)[["window_sec","acc","f1","auc","f1_std"]].round(3)
ablation_window.to_csv(os.path.join(OUT_DIR, "ablation_window.csv"), index=False)
ablation_window

In [ ]:
# 8. Figures (saved to OUT_DIR for use as paper figures)
import matplotlib.pyplot as plt

# Fig 1: model comparison (macro-F1)
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(results.model, results.f1)
ax.set_ylabel("Macro-F1 (LOSO)"); ax.set_ylim(0,1); ax.axhline(0.5, ls="--", c="gray", lw=1)
plt.xticks(rotation=20, ha="right"); plt.title("Stress detection — LOSO")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_models.png", dpi=200); plt.show()

# Fig 2: per-subject F1 spread for Random Forest (generalization story)
fig, ax = plt.subplots(figsize=(5,4))
ax.boxplot(per_model_f1["Random Forest"], labels=["Random Forest"])
ax.scatter(np.ones(len(per_model_f1["Random Forest"])), per_model_f1["Random Forest"], alpha=0.6)
ax.set_ylabel("Per-subject macro-F1"); ax.set_ylim(0,1); plt.title("Per-subject variation (LOSO)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_per_subject.png", dpi=200); plt.show()

# Fig 3: Random Forest feature importance
sc = StandardScaler().fit(df[FEATURES])
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(sc.transform(df[FEATURES]), df.label)
order = np.argsort(rf.feature_importances_)
fig, ax = plt.subplots(figsize=(5,4))
ax.barh(np.array(FEATURES)[order], rf.feature_importances_[order])
ax.set_xlabel("Importance"); plt.title("RF feature importance")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_importance.png", dpi=200); plt.show()

In [ ]:
# 9. Build a results summary to paste into experimental_log.md (replaces PENDING)
lines = ["## 2. Results - Main (LOSO, baseline vs stress)", ""]
lines.append("| Model | Accuracy | Macro-F1 | AUROC | Per-subject F1 std |")
lines.append("|---|---|---|---|---|")
for _, r in results.iterrows():
    lines.append(f"| {r['model']} | {r['acc']:.3f} | {r['f1']:.3f} | {r['auc']:.3f} | {r['f1_std']:.3f} |")
lines += ["", "## 3. Ablations", "", "Feature groups (Random Forest):"]
for _, r in ablation_feat.iterrows():
    lines.append(f"- {r['features']}: acc={r['acc']:.3f}, F1={r['f1']:.3f}, AUROC={r['auc']:.3f}")
lines.append("Window length (Random Forest):")
for _, r in ablation_window.iterrows():
    lines.append(f"- {int(r['window_sec'])}s: acc={r['acc']:.3f}, F1={r['f1']:.3f}, AUROC={r['auc']:.3f}")
summary = "\n".join(lines)
with open(f"{OUT_DIR}/results_summary.md", "w") as fh:
    fh.write(summary)
print(summary)

## Next steps
1. Download the `outputs/` folder (CSVs + figures + `results_summary.md`).
2. Paste `results_summary.md` into `experimental_log.md`, replacing the `PENDING` tables
   with these **real** numbers, and copy the figures into the paper's `figures/` folder.
3. **Apple Watch Ultra validation (separate, local):** export Apple Health HRV (SDNN) +
   your self-reported stress labels, window them the same way, apply the trained model,
   and report agreement. Keep that personal data on your own machine.

Only after `experimental_log.md` holds real results do we run PaperOrchestra (or draft
the IEEE paper directly).
